# Day 8 · Exercise 5: Batch Support-Ticket Classifier

**What you'll build:** `classify_batch(tickets: list[str], labels: list[str], model: str) -> list[dict]` — a function that iterates over a list of support-ticket strings, classifies each one with a single-item `classify()` call, catches per-item exceptions without halting the loop, and collects every outcome into a list of result dicts with keys `text`, `label`, `status`, and `error`.

**Why it matters:** Real data pipelines process hundreds of items at a time — mastering the error-safe batch loop with structured result dicts and logging is what separates a brittle script that crashes on item 47 from a production classifier you can trust to run unattended overnight.

## Your Implementation

In [ ]:
import logging
import ollama

# Configure logging once here at the entry point — NOT inside the function.
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s  %(levelname)-8s  %(name)s  %(message)s",
)

logger = logging.getLogger(__name__)

# ── Single-item classifier (already provided — do not modify) ──────────────
_SYSTEM_PROMPT = (
    "You are a support-ticket classifier. "
    "The possible labels are: {labels}. "
    "Reply with exactly one label. No explanation. No punctuation."
)


def classify(text: str, labels: list[str], model: str) -> str:
    """Classify a single text string and return exactly one label.

    Args:
        text:   The support ticket text to classify.
        labels: The full set of allowed label strings.
        model:  Ollama model name.

    Returns:
        A single lowercase label string that is a member of labels.

    Raises:
        ValueError: If the model returns a string not in labels.
    """
    system = _SYSTEM_PROMPT.format(labels=", ".join(labels))
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": text},
        ],
    )
    raw: str = response["message"]["content"].strip().lower()
    if raw not in [lb.lower() for lb in labels]:
        raise ValueError(f"Model returned unexpected label: {raw!r}")
    return raw


# ── Your task: implement classify_batch below ──────────────────────────────

def classify_batch(tickets: list[str], labels: list[str], model: str) -> list[dict]:
    """Classify a list of support tickets, collecting results without stopping on errors.

    Loops over every ticket in tickets, calls classify() for each one, and
    appends one result dict to the output list regardless of success or failure.
    A per-item exception is caught, logged as a WARNING, and recorded as an
    error dict — it does NOT stop the loop or propagate to the caller.

    Logging contract (use the module-level logger):
      - logger.info()  at batch start: number of items and model name.
      - logger.debug() per item: index (1-based), total, and a 60-char text preview.
      - logger.info()  on success: index, total, and the label assigned.
      - logger.warning() on failure: index, total, a 40-char text preview, and the error.
      - logger.info()  at batch end: succeeded count, total, failed count.

    Args:
        tickets: List of support-ticket strings to classify.
        labels:  Full set of allowed label strings (e.g. ["billing", "technical-issue"]).
        model:   Ollama model name (e.g. "llama3.2").

    Returns:
        A list of dicts, one per input ticket, in the same order as tickets.
        Each dict always contains exactly these four keys:
          - "text":   str  — the original ticket string.
          - "label":  str | None — the classification result, or None on failure.
          - "status": str  — "ok" on success, "error" on failure.
          - "error":  str | None — None on success, the error message string on failure.

    Example:
        >>> results = classify_batch(
        ...     ["Charged twice", "App keeps crashing"],
        ...     labels=["billing", "technical-issue", "general"],
        ...     model="llama3.2",
        ... )
        >>> results[0]["status"]
        'ok'
        >>> results[0]["label"]
        'billing'
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 5 automated checks and shows ✅ / ❌ for each.

In [ ]:
import logging
import unittest.mock as mock

_PASS, _FAIL = '✅', '❌'

_LABELS = ["billing", "technical-issue", "feature-request", "shipping", "account-access", "general"]
_MODEL  = "llama3.2"

_TICKETS = [
    "I was charged twice for my subscription this month.",
    "The login page keeps crashing on Chrome and Firefox.",
    "Could you add dark mode to the dashboard?",
]
# This ticket will be injected to trigger an error result via a mock.
_BAD_TICKET = "__FORCE_ERROR__"


def _run_checks():
    score, total = 0, 5

    # ── Check 1: function exists and is callable ───────────────────────────
    try:
        assert callable(classify_batch), 'classify_batch is not defined or not callable'
        print(f'{_PASS} Check 1/{total}: classify_batch is defined and callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return  # Cannot continue — later checks would NameError

    # ── Check 2: returns a list with one dict per ticket ───────────────────
    # Mock classify() to return a deterministic label without hitting Ollama.
    def _fake_classify(text, labels, model):
        return "billing"

    try:
        with mock.patch(__name__ + '.classify', side_effect=_fake_classify):
            results = classify_batch(_TICKETS, _LABELS, _MODEL)
        assert isinstance(results, list), f'expected list, got {type(results).__name__}'
        assert len(results) == len(_TICKETS), (
            f'expected {len(_TICKETS)} results, got {len(results)}'
        )
        print(f'{_PASS} Check 2/{total}: returns a list with one dict per input ticket')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')
        return  # Dict-key checks below would fail unpredictably

    # ── Check 3: every success dict has all four required keys ────────────
    try:
        required_keys = {"text", "label", "status", "error"}
        for i, r in enumerate(results):
            assert isinstance(r, dict), f'result[{i}] is {type(r).__name__}, expected dict'
            missing = required_keys - r.keys()
            assert not missing, f'result[{i}] missing keys: {missing}'
            assert r["status"] == "ok",    f'result[{i}]["status"] = {r["status"]!r}, expected "ok"'
            assert r["error"]  is None,    f'result[{i}]["error"] should be None on success'
            assert r["label"]  is not None, f'result[{i}]["label"] should not be None on success'
            assert r["text"]   == _TICKETS[i], f'result[{i}]["text"] does not match input'
        print(f'{_PASS} Check 3/{total}: every result dict has all four required keys with correct values')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # ── Check 4: error isolation — one bad item → error dict, loop continues
    try:
        call_count = 0

        def _fake_classify_with_error(text, labels, model):
            nonlocal call_count
            call_count += 1
            if text == _BAD_TICKET:
                raise ValueError("forced test error")
            return "general"

        mixed = [_TICKETS[0], _BAD_TICKET, _TICKETS[1]]
        with mock.patch(__name__ + '.classify', side_effect=_fake_classify_with_error):
            mixed_results = classify_batch(mixed, _LABELS, _MODEL)

        assert call_count == 3, (
            f'classify() was called {call_count} time(s); expected 3 — loop stopped early'
        )
        assert len(mixed_results) == 3, (
            f'expected 3 result dicts, got {len(mixed_results)}'
        )
        err_result = mixed_results[1]
        assert err_result["status"] == "error", (
            f'bad item should have status "error", got {err_result["status"]!r}'
        )
        assert err_result["label"] is None, (
            f'bad item label should be None, got {err_result["label"]!r}'
        )
        assert err_result["error"] is not None, (
            'bad item error field should be a non-None string'
        )
        assert mixed_results[0]["status"] == "ok", 'item before error should still be ok'
        assert mixed_results[2]["status"] == "ok", 'item after error should still be ok'
        print(f'{_PASS} Check 4/{total}: one failing item records an error dict without stopping the loop')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    # ── Check 5: logger was called during the batch ────────────────────────
    try:
        with mock.patch(__name__ + '.classify', side_effect=_fake_classify):
            with mock.patch.object(logger, 'info') as mock_info, \
                 mock.patch.object(logger, 'debug') as mock_debug:
                classify_batch(_TICKETS, _LABELS, _MODEL)

        assert mock_info.call_count  >= 2, (
            f'logger.info() called {mock_info.call_count} time(s); '
            f'expected at least 2 (batch start + batch end)'
        )
        assert mock_debug.call_count >= len(_TICKETS), (
            f'logger.debug() called {mock_debug.call_count} time(s); '
            f'expected at least {len(_TICKETS)} (one per item)'
        )
        print(f'{_PASS} Check 5/{total}: logger.info() and logger.debug() are called during the batch')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 5/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
        print(f'  {total}/{total} passed.')
    else:
        print(f'  {score}/{total} passed. Keep going!')


_run_checks()

## Bonus Challenge

Right now `classify_batch` processes items one by one in a serial loop. Each `classify()` call blocks until the model responds, so ten tickets take roughly ten times as long as one.

Try adding a `max_retries` parameter: if `classify()` raises a `ValueError` (invalid label), retry that item up to `max_retries` times before recording it as an error. A simple inner loop works:

```python
def classify_batch(tickets, labels, model, max_retries=2):
    results = []
    for i, text in enumerate(tickets):
        for attempt in range(1, max_retries + 1):
            try:
                label = classify(text, labels, model)
                results.append({"text": text, "label": label, "status": "ok", "error": None})
                break  # success — exit the retry loop
            except ValueError as e:
                if attempt == max_retries:
                    results.append({"text": text, "label": None, "status": "error", "error": str(e)})
    return results
```

This foreshadows the resilience patterns you will build in a later section on error handling (Day 31), where you will add exponential backoff for external APIs that impose rate limits.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
def classify_batch(tickets: list[str], labels: list[str], model: str) -> list[dict]:
    results: list[dict] = []
    total = len(tickets)
    logger.info("classify_batch starting: %d items, model=%r", total, model)

    for i, text in enumerate(tickets):
        logger.debug("Item %d/%d: %r", i + 1, total, text[:60])
        try:
            label = classify(text, labels, model)
            logger.info("Item %d/%d -> %r", i + 1, total, label)
            results.append({
                "text": text,
                "label": label,
                "status": "ok",
                "error": None,
            })
        except Exception as e:
            logger.warning(
                "Item %d/%d failed (%r): %s", i + 1, total, text[:40], e
            )
            results.append({
                "text": text,
                "label": None,
                "status": "error",
                "error": str(e),
            })

    succeeded = sum(1 for r in results if r["status"] == "ok")
    failed = total - succeeded
    logger.info(
        "classify_batch done: %d/%d succeeded, %d failed",
        succeeded, total, failed,
    )
    return results
```

**Why this works:** The `try/except Exception` block inside the `enumerate` loop is the key design decision — it turns any per-item failure into data (an error dict) instead of a crash, so the loop always runs to completion. Storing `str(e)` as the `error` field preserves the failure reason without letting an exception object escape into a plain Python dict. The five log calls (info at start, debug and info/warning per item, info at end) follow the Day-5 pattern of using `%`-style format strings, which avoids the cost of string interpolation when the log level is disabled.
</details>